In [1]:
import MDAnalysis as mda

origin_structure = mda.Universe("proteins/3dd5.pdb")
ligand_mda = origin_structure.select_atoms("resname DEP")

In [2]:
pocket_center = ligand_mda.center_of_geometry()
pocket_center

array([  0.84715631, -11.80328124,  35.55646872])

In [3]:
ligand_box = ligand_mda.positions.max(axis=0) - ligand_mda.positions.min(axis=0) + 5
ligand_box

array([78.36 , 66.335, 90.642], dtype=float32)

In [4]:
pocket_center = pocket_center.tolist()
ligand_box = ligand_box.tolist()

In [5]:
import os

PDB_ID = "3dd5"
LIGAND = "dep"

os.makedirs("results", exist_ok=True)

In [6]:
from vina import Vina

In [7]:
v = Vina()
v.set_receptor("pdbqt/3dd5.pdbqt")
v.set_ligand_from_file("pdbqt/dep.pdbqt")
v.compute_vina_maps(center=pocket_center, box_size=ligand_box)

Computing Vina grid ... 

done.


In [8]:
v.dock(exhaustiveness=10, n_poses=10)

Performing docking (random seed: 1225722778) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -3.241          0          0
   2       -3.131      27.67      29.01
   3       -3.092      52.94      53.61
   4       -3.081      13.54      14.23
   5       -3.078      1.009      3.306
   6       -3.073      88.29      89.32
   7        -3.07      27.42       28.8
   8       -3.068      25.08      26.11
   9       -3.061      2.821      4.659
  10       -3.055      87.63      88.86


In [9]:
v.write_poses(f"results/{LIGAND}.pdbqt", n_poses=10, overwrite=True)

In [10]:
v.energies()

array([[-3.241, -3.999, -0.163,  0.758, -0.163],
       [-3.131, -3.94 , -0.086,  0.732, -0.163],
       [-3.092, -3.885, -0.093,  0.723, -0.163],
       [-3.081, -3.759, -0.205,  0.721, -0.163],
       [-3.078, -3.844, -0.117,  0.72 , -0.163],
       [-3.073, -3.8  , -0.154,  0.719, -0.163],
       [-3.07 , -3.89 , -0.06 ,  0.718, -0.163],
       [-3.068, -3.699, -0.25 ,  0.718, -0.163],
       [-3.061, -3.845, -0.094,  0.716, -0.163]])

In [11]:
import pandas as pd


# These are the columns for the types of energies according to AutoDock Vina docs.
column_names = ["total", "inter", "intra", "torsions", "intra best pose"]

df = pd.DataFrame(v.energies(), columns=column_names)
df

,total,inter,intra,torsions,intra best pose
0,-3.241,-3.999,-0.163,0.758,-0.163
1,-3.131,-3.940,-0.086,0.732,-0.163
2,-3.092,-3.885,-0.093,0.723,-0.163
3,-3.081,-3.759,-0.205,0.721,-0.163
4,-3.078,-3.844,-0.117,0.720,-0.163
5,-3.073,-3.800,-0.154,0.719,-0.163
6,-3.070,-3.890,-0.060,0.718,-0.163
7,-3.068,-3.699,-0.250,0.718,-0.163
8,-3.061,-3.845,-0.094,0.716,-0.163


In [12]:
df.to_csv("results/dep.csv", index=None)

In [13]:
! mk_export.py results/dep.pdbqt -s results/dep.sdf